# Los primeros modelos de GrantMatch

La preparación y la división se importan de [`pipeline_proyecto.py`](./pipeline_proyecto.py), no se copian. Es clasificación: el par (award, publicación) es un financiamiento real o no. Métrica: **AUC** (área bajo la curva ROC), que no depende de un umbral; también se reporta la exactitud. Al final, la tabla queda en `resultados_proyecto.csv`.

Antes de hacer commit: **Restart** y **Run All**, para que la tabla salga de una corrida completa.

In [ ]:
import time

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from pipeline_proyecto import anio_de_corte, cargar_y_dividir, construir_preparacion, RUTA_DATOS

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
print("Ano de corte:", anio_de_corte(pd.read_csv(RUTA_DATOS)))
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

In [ ]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    """Entrena el modelo despues de la preparacion y mide la AUC en
    entrenamiento y en prueba. Si vuelves a correr la celda de un modelo,
    su fila se reemplaza."""
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    auc_ent = roc_auc_score(y_ent, pipeline.predict_proba(X_ent)[:, 1])
    auc_prueba = roc_auc_score(y_prueba, pipeline.predict_proba(X_prueba)[:, 1])
    resultados[nombre] = {
        "modelo": nombre,
        "auc_entrenamiento": round(auc_ent, 4),
        "auc_prueba": round(auc_prueba, 4),
        "brecha": round(auc_ent - auc_prueba, 4),
        "exactitud_prueba": round(accuracy_score(y_prueba, pipeline.predict(X_prueba)), 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:30s} AUC entrenamiento {auc_ent:.4f}   prueba {auc_prueba:.4f}   {segundos:6.2f} s")

## Línea base

`DummyClassifier(strategy="prior")`: predice siempre la proporción de pares reales del entrenamiento. Con clases balanceadas su AUC es 0.5, lo que saca cualquier modelo que no distinga un par real de uno al azar.

In [ ]:
evaluar("linea base", DummyClassifier(strategy="prior"))

## Modelo lineal: regresión logística, con la regularización por defecto y una más fuerte

In [ ]:
evaluar("logistica C=1", LogisticRegression(C=1.0, max_iter=1000))

In [ ]:
evaluar("logistica C=0.01", LogisticRegression(C=0.01, max_iter=1000))

## Ensamblajes: un bosque sin límite, el mismo con profundidad limitada, y gradient boosting

In [ ]:
evaluar("bosque sin limite", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))

In [ ]:
evaluar("bosque max_depth=8", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))

In [ ]:
evaluar("gradient boosting", GradientBoostingClassifier(random_state=42))

## Tabla de resultados

In [ ]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("resultados_proyecto.csv", index=False)
mejor = tabla.loc[tabla["auc_prueba"].idxmax()]
print(f"Mejor en prueba: {mejor['modelo']} (AUC {mejor['auc_prueba']:.4f}, "
      f"{mejor['auc_prueba'] - resultados['linea base']['auc_prueba']:+.4f} sobre la linea base)")
tabla